In [1]:
import pandas as pd
from transformers import AutoTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from transformers import TrainingArguments
from transformers import AutoModelForSequenceClassification
import torch

c:\Users\easyl\Documents\My_code\GitHub\RuReviews-sentiment\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Загрузка и предобработка датасета

In [2]:
from datasets import load_dataset
dataset = load_dataset('mteb/RuReviewsClassification')

In [ ]:
print(dataset)
print(dataset['train'].features)

for label in [0, 1, 2]:
    print(f"=== Label {label} ===")
    examples = [x for x in dataset['train'] if x['label'] == label][:8]
    for elem in examples:
        print(elem['text'][:200])
        print('---')
    print()

In [3]:
label_names = {0: 'negative', 1: 'neutral', 2: 'positive'}

In [4]:
labels = dataset['train']['label']
counts = pd.Series(labels).value_counts().sort_index()
counts.index = counts.index.map(label_names)
print(counts)

negative    15000
neutral     15000
positive    15000
Name: count, dtype: int64


## Токенизация данных

In [13]:
tokenizer = AutoTokenizer.from_pretrained('cointegrated/rubert-tiny2')

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 2048/2048 [00:00<00:00, 30563.40 examples/s]


In [14]:
print(tokenized_dataset)
print(tokenized_dataset['train'][0].keys())
print(len(tokenized_dataset['train'][0]['input_ids']))

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 45000
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 15000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2048
    })
})
dict_keys(['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])
128


## Baseline: TF-IDF + LogisticRegression

In [16]:
train_texts = dataset['train']['text']
train_labels = dataset['train']['label']

test_texts = dataset['test']['text']
test_labels = dataset['test']['label']

vectorizer = TfidfVectorizer(max_features=10000)
X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)

baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train, train_labels)

baseline_preds = baseline_model.predict(X_test)
baseline_f1 = f1_score(test_labels, baseline_preds, average='macro')

print(f"Baseline F1 (macro): {baseline_f1:.4f}")
print(classification_report(test_labels, baseline_preds, target_names=['negative', 'neutral', 'positive']))

Baseline F1 (macro): 0.7353
              precision    recall  f1-score   support

    negative       0.72      0.70      0.71       682
     neutral       0.62      0.64      0.63       683
    positive       0.87      0.86      0.86       683

    accuracy                           0.73      2048
   macro avg       0.74      0.73      0.74      2048
weighted avg       0.74      0.73      0.74      2048



In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    'cointegrated/rubert-tiny2',
    num_labels=3
)

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 36698.49it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the c

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_strategy='steps',
    logging_steps=100,
)


In [33]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    f1 = f1_score(labels, predictions, average='macro')
    accuracy = accuracy_score(labels, predictions)
    return {'f1': f1, 'accuracy': accuracy}

In [34]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics,
)

In [35]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.584700,0.568696,0.749042,0.749067
2,0.545885,0.553009,0.761661,0.759333
3,0.494421,0.556426,0.763069,0.761600


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.74it/s]


TrainOutput(global_step=4221, training_loss=0.5592006560440623, metrics={'train_runtime': 70.1005, 'train_samples_per_second': 1925.808, 'train_steps_per_second': 60.214, 'total_flos': 248911937280000.0, 'train_loss': 0.5592006560440623, 'epoch': 3.0})

In [36]:
from transformers import TrainingArguments, EarlyStoppingCallback

training_args_v2 = TrainingArguments(
    output_dir='./results_v2',
    num_train_epochs=8,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=200,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_strategy='steps',
    logging_steps=100,
    save_total_limit=2,
)

model_v2 = AutoModelForSequenceClassification.from_pretrained(
    'cointegrated/rubert-tiny2',
    num_labels=3
)

trainer_v2 = Trainer(
    model=model_v2,
    args=training_args_v2,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer_v2.train()

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 54925.41it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the c

Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.597386,0.583442,0.744444,0.744067
2,0.560506,0.556583,0.757314,0.754733
3,0.507336,0.556822,0.759849,0.756533
4,0.494900,0.549585,0.762565,0.759733
5,0.474164,0.561189,0.762528,0.760600
6,0.462361,0.565579,0.762956,0.760933
7,0.443337,0.575230,0.761628,0.760800
8,0.411593,0.573264,0.763193,0.761533


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.01it/s]


TrainOutput(global_step=11256, training_loss=0.5019464136949227, metrics={'train_runtime': 186.2257, 'train_samples_per_second': 1933.138, 'train_steps_per_second': 60.443, 'total_flos': 663765166080000.0, 'train_loss': 0.5019464136949227, 'epoch': 8.0})

In [37]:
lengths = [len(tokenizer.encode(text)) for text in dataset['train']['text'][:2000]]
import numpy as np
print(f"Среднее: {np.mean(lengths):.1f}")
print(f"95-й перцентиль: {np.percentile(lengths, 95):.1f}")
print(f"Максимум: {np.max(lengths)}")

Среднее: 34.1
95-й перцентиль: 90.0
Максимум: 252


In [38]:
import torch.nn as nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Увеличиваем вес neutral класса (индекс 1)
        class_weights = torch.tensor([1.0, 1.5, 1.0]).to(logits.device)
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss

In [39]:
model_v3 = AutoModelForSequenceClassification.from_pretrained(
    'cointegrated/rubert-tiny2',
    num_labels=3
)

training_args_v3 = TrainingArguments(
    output_dir='./results_v3',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_strategy='steps',
    logging_steps=100,
)

trainer_v3 = WeightedTrainer(
    model=model_v3,
    args=training_args_v3,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics,
)

trainer_v3.train()

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 36651.85it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the c

Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.590599,0.580447,0.752050,0.748200
2,0.551777,0.562679,0.756107,0.752067
3,0.501291,0.566927,0.758660,0.754667


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.81it/s]


TrainOutput(global_step=4221, training_loss=0.568876327933985, metrics={'train_runtime': 71.3688, 'train_samples_per_second': 1891.582, 'train_steps_per_second': 59.143, 'total_flos': 248911937280000.0, 'train_loss': 0.568876327933985, 'epoch': 3.0})

## Заменим модель

In [40]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer_base = AutoTokenizer.from_pretrained('DeepPavlov/rubert-base-cased')
model_base = AutoModelForSequenceClassification.from_pretrained(
    'DeepPavlov/rubert-base-cased',
    num_labels=3
)

c:\Users\easyl\Documents\My_code\GitHub\RuReviews-sentiment\venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\easyl\.cache\huggingface\hub\models--DeepPavlov--rubert-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 56876.76it/s]
[tran

In [41]:
def tokenize_function_base(examples):
    return tokenizer_base(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=128
    )

tokenized_dataset_base = dataset.map(tokenize_function_base, batched=True)

Map: 100%|██████████| 2048/2048 [00:00<00:00, 28837.96 examples/s]


In [42]:
training_args_base = TrainingArguments(
    output_dir='./results_base',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_strategy='steps',
    logging_steps=100,
)

In [43]:
trainer_base = Trainer(
    model=model_base,
    args=training_args_base,
    train_dataset=tokenized_dataset_base['train'],
    eval_dataset=tokenized_dataset_base['validation'],
    compute_metrics=compute_metrics,
)

trainer_base.train()

Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.550080,0.530860,0.772471,0.769400
2,0.470160,0.549135,0.776068,0.772800
3,0.372575,0.598158,0.770178,0.767733


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.06s/it]


TrainOutput(global_step=8439, training_loss=0.4738363356849624, metrics={'train_runtime': 652.5734, 'train_samples_per_second': 206.873, 'train_steps_per_second': 12.932, 'total_flos': 8880077848320000.0, 'train_loss': 0.4738363356849624, 'epoch': 3.0})

## Переход к LLM (Zero-shot/few-shot без обучения)

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer_llm = AutoTokenizer.from_pretrained(model_name)
model_llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

W0919 05:52:40.112000 26824 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|██████████| 434/434 [00:02<00:00, 171.63it/s]


In [14]:
def classify_sentiment_llm(text):
    prompt = f"""Определи тональность следующего отзыва на товар. Ответь только одним словом: negative, neutral или positive.

Отзыв: {text}

Тональность:"""

    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer_llm.apply_chat_template(
        messages, 
        tokenize=True, 
        add_generation_prompt=True, 
        return_tensors="pt",
        return_dict=True
    ).to(model_lora.device)

    outputs = model_lora.generate(**inputs, max_new_tokens=10, do_sample=False)
    response = tokenizer_llm.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip().lower()

In [7]:
sample = dataset['test'][0]['text']
print(f"Текст: {sample}")
print(f"Ответ модели: {classify_sentiment_llm(sample)}")

Текст: Это не платье,это длинная бесформенная майка.Хотела в этом платье встречать Новый год,а пришлось применить его как ночнушку.Товар несоответствие картинке,не рекомендую.
Ответ модели: negative


In [8]:
import random

test_indices = random.sample(range(len(dataset['test'])), 10)

for idx in test_indices:
    text = dataset['test'][idx]['text']
    true_label = dataset['test'][idx]['label']
    true_label_name = label_names[true_label]
    
    predicted = classify_sentiment_llm(text)
    
    print(f"Текст: {text[:100]}...")
    print(f"Истинная метка: {true_label_name} | Предсказание модели: {predicted}")
    print("---")

Текст: Брак....
Истинная метка: negative | Предсказание модели: negative
---
Текст: заказала на свой 44 размер L   т.е  на размер больше. получила маленькую рубашку размера 40-42. очен...
Истинная метка: negative | Предсказание модели: negative
---
Текст: Завязки пришиты криво, цвет оказался голубее, чем на фото....
Истинная метка: neutral | Предсказание модели: negative
---
Текст: Very upset. Skirt do not match the description. To wear It is not possible...
Истинная метка: negative | Предсказание модели: negative
---
Текст: Пришла посылка с принадлежностями для кухни, по данному треку. Похоже что-то перепутано, но мне от э...
Истинная метка: negative | Предсказание модели: negative
---
Текст: Размерная сетка не совпадает и крой плохой ...
Истинная метка: negative | Предсказание модели: negative
---
Текст: Размер в размер! МЕГА крутое платье! Продавцу спасибо! ...
Истинная метка: positive | Предсказание модели: positive
---
Текст: Заказала розовую юбку, цвет грязный, совершенно другой,

In [9]:
def format_instruction(example):
    label_name = label_names[example['label']]
    prompt = f"""Определи тональность следующего отзыва на товар. Ответь только одним словом: negative, neutral или positive.

Отзыв: {example['text']}

Тональность:"""
    full_text = prompt + " " + label_name
    return {"text": full_text}

formatted_dataset = dataset.map(format_instruction)
print(formatted_dataset['train'][0]['text'])

Определи тональность следующего отзыва на товар. Ответь только одним словом: negative, neutral или positive.

Отзыв: всё пришло спасибо. только немного короче чем я ожидала
так всё супер

Тональность: positive


In [10]:
def tokenize_for_causal_lm(examples):
    tokenized = tokenizer_llm(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=128
    )
    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized

tokenized_causal_dataset = formatted_dataset.map(tokenize_for_causal_lm, batched=True)

In [58]:
print(tokenized_causal_dataset['train'][0].keys())
print(len(tokenized_causal_dataset['train'][0]['input_ids']))

dict_keys(['text', 'label', 'input_ids', 'attention_mask', 'labels'])
128


In [11]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

model_llm = prepare_model_for_kbit_training(model_llm)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)

model_lora = get_peft_model(model_llm, lora_config)
model_lora.print_trainable_parameters()

trainable params: 1,843,200 || all params: 3,087,781,888 || trainable%: 0.0597


In [12]:
from transformers import TrainingArguments, Trainer

training_args_lora = TrainingArguments(
    output_dir='./results_lora',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=20,
    bf16=True,
)

trainer_lora = Trainer(
    model=model_lora,
    args=training_args_lora,
    train_dataset=tokenized_causal_dataset['train'].select(range(2000)),
    eval_dataset=tokenized_causal_dataset['validation'].select(range(200)),
)

In [13]:
trainer_lora.train()

c:\Users\easyl\Documents\My_code\GitHub\RuReviews-sentiment\venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,1.003015,0.960540


TrainOutput(global_step=125, training_loss=1.410846700668335, metrics={'train_runtime': 311.6287, 'train_samples_per_second': 6.418, 'train_steps_per_second': 0.401, 'total_flos': 4264883650560000.0, 'train_loss': 1.410846700668335, 'epoch': 1.0})

In [15]:
import random

random.seed(42)  # для воспроизводимости, чтобы сравнить с похожей выборкой
test_indices = random.sample(range(len(dataset['test'])), 15)

correct = 0
for idx in test_indices:
    text = dataset['test'][idx]['text']
    true_label = dataset['test'][idx]['label']
    true_label_name = label_names[true_label]
    
    predicted = classify_sentiment_llm(text)
    is_correct = predicted == true_label_name
    correct += is_correct
    
    print(f"Текст: {text[:80]}...")
    print(f"Истина: {true_label_name} | Предсказание: {predicted} | {'✅' if is_correct else '❌'}")
    print("---")

print(f"\nAccuracy на выборке: {correct}/{len(test_indices)} = {correct/len(test_indices)*100:.1f}%")

Текст: Великолепные плотные носочки, целых 5 пар! На фото всего 4, потому что бежевые у...
Истина: positive | Предсказание: positive | ✅
---
Текст: Все здорово ...
Истина: positive | Предсказание: positive | ✅
---
Текст: Цена соответствует качеству. Оочень большое декольте, на фото оно тоже представл...
Истина: neutral | Предсказание: negative | ❌
---
Текст: Я не получила свой заказ за три с половиной месяца.Открыла спор.Деньги вернули с...
Истина: negative | Предсказание: negative | ✅
---
Текст: очень сильно маломерит, не соответствует заявленным размерам. осталась не доволь...
Истина: negative | Предсказание: negative | ✅
---
Текст: ооочень крутые лосики!!! рекомендую...
Истина: positive | Предсказание: positive | ✅
---
Текст: дырка по шву с внутренней стороны
 шапки...
Истина: negative | Предсказание: negative | ✅
---
Текст: трек не отслеживался, но товар пришел очень быстро. качество отличное...
Истина: positive | Предсказание: positive | ✅
---
Текст: Ткань ужасная. Похожа на ночно